In [1]:
import os
import sys
import glob
import time

import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm

# --- FIXED PARAMETERS ---
MODEL_PATH = "/Users/angelinacu/Desktop/Study/Viettel/IDENTIFY-THE-PARCEL-AT-THE-TOP-OF-THE-CONVEYOR-BELT/source_haanh/my_model.pt"
IMG_SOURCE_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Test/rgb"
SAVE_DIR = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/yolo_center" # Directory to save the output .txt files
CONF_THRESHOLD = 0.5 # Minimum confidence threshold to consider a detection
ROI = (560, 150, 300, 330) # ROI coordinates (x_min, y_min, width, height)
TARGET_CLASS = 'packet' # Only save results for this class
# --- END FIXED PARAMETERS ---

# Check if model file exists
if not os.path.exists(MODEL_PATH):
    print(f'ERROR: Model path is invalid or model was not found: {MODEL_PATH}')
    sys.exit(0)

# Load the model
try:
    model = YOLO(MODEL_PATH, task='detect')
    labels = model.names
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    sys.exit(0)

# Check if image source directory exists
if not os.path.isdir(IMG_SOURCE_DIR):
    print(f'ERROR: Image source directory not found: {IMG_SOURCE_DIR}')
    sys.exit(0)

# Get list of image files
img_ext_list = ['.jpg','.JPG','.jpeg','.JPEG','.png','.PNG','.bmp','.BMP']
imgs_list = []
filelist = glob.glob(os.path.join(IMG_SOURCE_DIR, '*'))
for file in filelist:
    _, file_ext = os.path.splitext(file)
    if file_ext in img_ext_list:
        imgs_list.append(file)

if not imgs_list:
    print(f"ERROR: No images found in directory: {IMG_SOURCE_DIR}")
    sys.exit(0)

print(f"Found {len(imgs_list)} images to process.")

# Create save directory if it doesn't exist
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Results will be saved in: {SAVE_DIR}")

# --- Inference Loop ---
processed_count = 0
for img_filename in tqdm(imgs_list, desc="Processing images"):
    try:
        frame = cv2.imread(img_filename)
        if frame is None:
            print(f"Warning: Could not read image {img_filename}. Skipping.")
            continue

        # Run inference on frame
        results = model(frame, verbose=False) # verbose=False speeds it up slightly

        # Extract results
        detections = results[0].boxes

        output_lines = []
        roi_x, roi_y, roi_w, roi_h = ROI # Unpack ROI

        # Go through each detection
        for i in range(len(detections)):
            # Get bounding box coordinates
            xyxy_tensor = detections[i].xyxy.cpu()
            xyxy = xyxy_tensor.numpy().squeeze()
            xmin, ymin, xmax, ymax = xyxy.astype(int)
            center_x = (xmin + xmax) / 2.0 # Use float division
            center_y = (ymin + ymax) / 2.0 # Use float division

            # Get class ID and name
            classidx = int(detections[i].cls.item())
            classname = labels.get(classidx, 'unknown') # Use .get for safety

            # Get confidence
            conf = detections[i].conf.item()

            # Check if detection is inside ROI
            is_inside_roi = (roi_x <= center_x <= roi_x + roi_w) and \
                            (roi_y <= center_y <= roi_y + roi_h)

            # Check confidence, ROI, and class name
            if conf >= CONF_THRESHOLD and is_inside_roi and classname == TARGET_CLASS:
                # Format the output line specifically for the next step
                # Format: classname confidence xmin xmax ymin ymax center_x center_y
                output_lines.append(f"{classname} {conf:.3f} {xmin} {xmax} {ymin} {ymax} {center_x:.1f} {center_y:.1f}")

        # Save results to a text file named after the image
        if output_lines:
            base_name = os.path.splitext(os.path.basename(img_filename))[0]
            # Ensure the output name matches the expected format (e.g., 0000.txt for image_0000.png)
            # Assuming image names are like 'image_XXXX.png'
            if base_name.startswith('image_'):
                 output_base_name = base_name.split('image_')[1] # Get 'XXXX'
            else:
                 output_base_name = base_name # Use original name if format differs

            txt_path = os.path.join(SAVE_DIR, f"{output_base_name}.txt")
            with open(txt_path, 'w') as f:
                f.write('\n'.join(output_lines))
            processed_count += 1
        # else:
            # print(f"No '{TARGET_CLASS}' detected above threshold {CONF_THRESHOLD} in ROI for {os.path.basename(img_filename)}")

    except Exception as e:
        print(f"Error processing image {img_filename}: {e}")

# --- End Loop ---

print("\nProcessing complete.")
print(f"Successfully processed and saved results for {processed_count} images.")

Model loaded successfully.
Found 20 images to process.
Results will be saved in: /Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/yolo_center


Processing images: 100%|██████████| 20/20 [00:02<00:00,  7.39it/s]


Processing complete.
Successfully processed and saved results for 20 images.


In [2]:
import pandas as pd
import numpy as np
import os
import cv2 # Thư viện OpenCV để đọc ảnh
import re
from tqdm import tqdm
import math

# --- HẰNG SỐ CẤU HÌNH ---
YOLO_RESULT_DIR = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/yolo_center"
DEPTH_IMG_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Test/depth"
OUTPUT_CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/submissions/Submission_3D.csv"

# --- Ngưỡng ---
HEIGHT_TOLERANCE_M = 0.005 # 5mm


# --- THAM SỐ CAMERA (Màu - Color) ---
color_intr = {
    "fx": 643.90087890625,
    "fy": 643.1365356445312,
    "cx": 650.2113037109375,
    "cy": 355.79559326171875,
    "coeffs": [-0.05658450722694397, 0.06544225662946701, -0.0008694113348610699, 0.00016751799557823688, -0.020957745611667633]
}
color_camera_matrix = np.array([
    [color_intr["fx"], 0, color_intr["cx"]],
    [0, color_intr["fy"], color_intr["cy"]],
    [0, 0, 1]
])
color_dist_coeffs = np.array(color_intr["coeffs"])

# --- THAM SỐ CAMERA (Độ sâu - Depth) ---
depth_intr = {
    "fx": 650.0616455078125,
    "fy": 650.0616455078125,
    "cx": 649.5928955078125,
    "cy": 360.9415588378906
}

def get_image_filename_from_txt(txt_filename):
    """Chuyển '0000.txt' thành 'image_0000.png'"""
    base_name = os.path.splitext(txt_filename)[0]
    return f"image_{base_name}.png"

def calculate_3d_point(u, v, depth_value_mm, intrinsics):
    """
    Chiếu ngược điểm ảnh 2D + độ sâu sang 3D (hệ tọa độ camera depth).
    """
    if depth_value_mm == 0:
        return None
    z_d = float(depth_value_mm) / 1000.0 # Chuyển mm sang m
    x_d = (u - intrinsics["cx"]) * z_d / intrinsics["fx"]
    y_d = (v - intrinsics["cy"]) * z_d / intrinsics["fy"]
    return np.array([x_d, y_d, z_d]) 

def main():

    results_list = []

    if not os.path.isdir(YOLO_RESULT_DIR):
        print(f"LỖI: Không tìm thấy thư mục YOLO: {YOLO_RESULT_DIR}")
        return
    
    yolo_files = [f for f in os.listdir(YOLO_RESULT_DIR) if f.endswith('.txt')]
    if not yolo_files:
        print(f"Không tìm thấy tệp .txt nào trong: {YOLO_RESULT_DIR}")
        return

    print(f"Tìm thấy {len(yolo_files)} tệp YOLO. Bắt đầu xử lý...")

    for yolo_filename in tqdm(yolo_files, desc="Processing images"):
        base_name = os.path.splitext(yolo_filename)[0]
        image_filename_for_csv = get_image_filename_from_txt(yolo_filename)
        yolo_filepath = os.path.join(YOLO_RESULT_DIR, yolo_filename)
        depth_filepath = os.path.join(DEPTH_IMG_DIR, f"{base_name}.png")

        if not os.path.exists(depth_filepath):
            print(f"Cảnh báo: Bỏ qua {yolo_filename} vì không tìm thấy ảnh depth: {depth_filepath}")
            continue

        detected_packets_3d = [] 

        try:
            depth_image = cv2.imread(depth_filepath, cv2.IMREAD_UNCHANGED)
            if depth_image is None: continue

            with open(yolo_filepath, 'r') as f: lines = f.readlines()
            if not lines: continue 

            for line in lines:
                parts = line.strip().split()
                if len(parts) != 8 or parts[0] != 'packet': continue

                try:
                    center_x_rgb_distorted = float(parts[6])
                    center_y_rgb_distorted = float(parts[7])
                    distorted_point = np.array([[[center_x_rgb_distorted, center_y_rgb_distorted]]], dtype=np.float32)

                    undistorted_normalized = cv2.undistortPoints(distorted_point, color_camera_matrix, color_dist_coeffs, P=color_camera_matrix)
                    u_undistorted = undistorted_normalized[0][0][0]
                    v_undistorted = undistorted_normalized[0][0][1]
                    u_lookup = int(round(u_undistorted))
                    v_lookup = int(round(v_undistorted))

                    if 0 <= v_lookup < depth_image.shape[0] and 0 <= u_lookup < depth_image.shape[1]:
                        depth_value_mm = depth_image[v_lookup, u_lookup]
                        point_3d = calculate_3d_point(u_lookup, v_lookup, depth_value_mm, depth_intr)
                        if point_3d is not None:
                            detected_packets_3d.append(point_3d)
                except ValueError: continue
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh {image_filename_for_csv}: {e}")
            continue

        if not detected_packets_3d: 
            print(f"Cảnh báo: Không tìm thấy packet hợp lệ nào trong {image_filename_for_csv}")
            continue 

        min_z = min(p[2] for p in detected_packets_3d)
        top_level_packets = [
            p for p in detected_packets_3d
            if p[2] <= min_z + HEIGHT_TOLERANCE_M
        ]

        if len(top_level_packets) == 1:
            topmost_packet_3d = top_level_packets[0]
        else:
            topmost_packet_3d = max(top_level_packets, key=lambda p: p[2])

        try:
            results_list.append({
                'image_filename': image_filename_for_csv,
                'x': topmost_packet_3d[0],
                'y': topmost_packet_3d[1],
                'z': topmost_packet_3d[2],
            })
        except Exception as e:
            print(f"Lỗi khi thêm kết quả của {image_filename_for_csv}: {e}")

    if not results_list:
        print("\nKhông có kết quả nào được phát hiện để lưu.")
        return

    print(f"\nĐã xử lý xong. Đang lưu {len(results_list)} kết quả vào CSV...")
    
    results_df = pd.DataFrame(results_list)
    
    columns_ordered = ['image_filename', 'x', 'y', 'z']
    results_df = results_df[columns_ordered]

    try:
        # --- [ĐÂY LÀ THAY ĐỔI] ---
        # Lưu vào file CSV, định dạng float 3 chữ số
        results_df.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.3f')
        print(f"Đã lưu thành công vào: {OUTPUT_CSV_PATH}")
    except Exception as e:
        print(f"\nLỖI khi lưu file CSV: {e}")


if __name__ == "__main__":
    main()

Tìm thấy 20 tệp YOLO. Bắt đầu xử lý...


Processing images: 100%|██████████| 20/20 [00:00<00:00, 90.71it/s]


Đã xử lý xong. Đang lưu 20 kết quả vào CSV...
Đã lưu thành công vào: /Users/angelinacu/Desktop/Study/Viettel/submissions/Submission_3D.csv


In [3]:
import pandas as pd
import open3d as o3d
import numpy as np
import os
import re  # Để trích xuất tên file
from tqdm import tqdm  # Để xem thanh tiến trình

# --- HẰNG SỐ CẤU HÌNH ---
# [SỬA] Cả đầu vào và đầu ra đều là cùng một file
CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/Code_Angie/result/Submission_3D.csv"
PLY_DIR = "/Users/angelinacu/Desktop/Study/Viettel/ThiSinh/Test/ply/"
OUTPUT_CSV_PATH = "/Users/angelinacu/Desktop/Study/Viettel/submissions/Submission_3D.csv"

# Vector pháp tuyến mặc định nếu tính toán thất bại
DEFAULT_NORMAL = np.array([0.0, 0.0, 1.0])

# Ma trận chuyển đổi 4x4 cho open3d
TRANSFORM_MATRIX = np.array([
    [1, 0, 0, 0],
    [0, -1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, 1]
])

# --- THAM SỐ THUẬT TOÁN (CỐ ĐỊNH) ---
K_NEIGHBORS = 4000
RANSAC_DISTANCE_THRESHOLD = 0.008
RANSAC_N = 3
RANSAC_ITERATIONS = 1000


def get_ply_path_from_image(image_filename, base_ply_dir):
    """
    Chuyển đổi tên file từ 'image_XXXX.png' thành đường dẫn '.../XXXX.ply'.
    """
    match = re.search(r'image_(\d+)\.png', image_filename)
    if not match:
        return None
    ply_name = f"{match.group(1)}.ply"
    return os.path.join(base_ply_dir, ply_name)

def calculate_normal_and_curvature(pcd, center_point, k, dist_thresh, n_pts, iters):
    """
    Tính normal vector VÀ surface variation (độ cong)
    ... (Nội dung hàm giữ nguyên) ...
    """
    try:
        pcd_tree = o3d.geometry.KDTreeFlann(pcd)
        [k_found, idx, _] = pcd_tree.search_knn_vector_3d(center_point, k)
        
        if k_found < n_pts:
            return None, None

        neighbor_points = np.asarray(pcd.points)[idx, :]
        pcd_neighbors = o3d.geometry.PointCloud()
        pcd_neighbors.points = o3d.utility.Vector3dVector(neighbor_points)

        plane_model, inlier_indices = pcd_neighbors.segment_plane(
            distance_threshold=dist_thresh,
            ransac_n=n_pts,
            num_iterations=iters
        )
        
        inlier_points = neighbor_points[inlier_indices, :]
        
        if inlier_points.shape[0] < n_pts:
            return None, None

        covariance_matrix = np.cov(inlier_points, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
        
        lambda_0 = eigenvalues[0]
        lambda_1 = eigenvalues[1]
        lambda_2 = eigenvalues[2]
        
        sum_eigenvalues = lambda_0 + lambda_1 + lambda_2
        
        if sum_eigenvalues == 0:
            return None, None 

        surface_variation = lambda_0 / sum_eigenvalues
        normal_vector = eigenvectors[:, 0] 
        
        return normal_vector, surface_variation

    except Exception as e:
        return None, None

# --- [SỬA] ---
# Đã xóa hàm compare_normals() vì không cần thiết khi chạy submission
# --- [KẾT THÚC SỬA] ---


def main():
    # 1. Đọc file CSV (chỉ chứa x, y, z)
    try:
        df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(f"LỖI: Không tìm thấy file CSV tại: {CSV_PATH}")
        return
    except KeyError:
        print(f"LỖI: File {CSV_PATH} dường như đã có cột Rx,Ry,Rz? Xóa file đó đi và chạy lại script 2D-to-3D trước.")
        return


    # Danh sách lưu kết quả cho file Submission (x,y,z,Rx,Ry,Rz)
    submission_list = []



    print(f"Bắt đầu tính toán Normal Vector (RANSAC+PCA) cho {len(df)} ảnh...")
    
    # 2. Lặp qua từng hàng trong file CSV (từng mẫu)
    for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Đang xử lý file"):
        image_name = row['image_filename']
        center_point = np.array([row['x'], row['y'], row['z']])
        
        
        # Đặt giá trị Rx, Ry, Rz mặc định
        final_normal = DEFAULT_NORMAL
        
        ply_path = get_ply_path_from_image(image_name, PLY_DIR)
        
        pcd = None
        if ply_path and os.path.exists(ply_path):
            try:
                pcd = o3d.io.read_point_cloud(ply_path)
                if pcd.is_empty():
                    pcd = None
                else:
                    pcd.transform(TRANSFORM_MATRIX)
            except Exception as e:
                pcd = None 

        # Chỉ tính toán nếu PCL được tải thành công
        if pcd is not None:
            calculated_normal, surface_variation = calculate_normal_and_curvature(
                pcd, 
                center_point, 
                K_NEIGHBORS,
                RANSAC_DISTANCE_THRESHOLD,
                RANSAC_N, 
                RANSAC_ITERATIONS
            )
            
            # Nếu tính toán thành công, cập nhật giá trị
            if calculated_normal is not None:
                final_normal = calculated_normal # Dùng vector đã tính
                
        
        # Luôn luôn thêm vào submission_list (dùng giá trị mặc định hoặc đã tính)
        submission_list.append({
            'image_filename': image_name,
            'x': center_point[0],
            'y': center_point[1],
            'z': center_point[2],
            'Rx': final_normal[0],
            'Ry': final_normal[1],
            'Rz': final_normal[2]
        })

    # --- KẾT THÚC VÒNG LẶP CHÍNH ---

    # 5. Tạo DataFrame và xuất file Submission
    if not submission_list:
        print("\nKhông xử lý thành công bất kỳ file nào.")
        return

    # A. Lưu file CSV submission
    results_df = pd.DataFrame(submission_list)
    
    # Sắp xếp lại cột để đảm bảo đúng thứ tự
    columns_ordered = ['image_filename', 'x', 'y', 'z', 'Rx', 'Ry', 'Rz']
    results_df = results_df[columns_ordered]
    
    # Ghi đè lên file OUTPUT_CSV_PATH (cũng là file CSV_PATH)
    results_df.to_csv(OUTPUT_CSV_PATH, index=False, float_format='%.3f')
    print(f"\nĐã tính toán và ghi đè 7 cột (đã làm tròn 3 số) vào file: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()

Bắt đầu tính toán Normal Vector (RANSAC+PCA) cho 20 ảnh...


Đang xử lý file: 100%|██████████| 20/20 [00:08<00:00,  2.44it/s]


Đã tính toán và ghi đè 7 cột (đã làm tròn 3 số) vào file: /Users/angelinacu/Desktop/Study/Viettel/submissions/Submission_3D.csv
